In [ ]:
#!/usr/bin/env python
"""
Notebook: 03_feature_engineering.ipynb
Feature Engineering for ECG Classification
"""

 # Feature Engineering for ECG Classification
# This notebook demonstrates feature extraction:
 - RR interval features (HRV)
 - Wavelet features
 - Morphological features (PQRST)

#  1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

sys.path.insert(0, '..')

from src.data.loader import ECGLoader
from src.data.preprocessor import ECGPreprocessor
from src.data.segmenter import ECGSegmenter
from src.features.rr_intervals import RRIntervalFeatures
from src.features.wavelet import WaveletFeatures
from src.features.morphological import MorphologicalFeatures
from src.features.fusion import FeatureFusion

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# 2. Load and Preprocess Data

In [ ]:
# Load a sample record
loader = ECGLoader(data_dir="../data/raw")
record = loader.load_mit_bih_record(100, leads=[0])

# Preprocess
preprocessor = ECGPreprocessor(sampling_rate=record.sampling_rate)
signal = preprocessor.preprocess(record.signal[:, 0])

# Detect R-peaks and segment beats
segmenter = ECGSegmenter(sampling_rate=record.sampling_rate)
r_peaks, _ = segmenter.detect_r_peaks(signal)
beats = segmenter.segment_beats(signal, r_peaks)

print(f"Processed {len(beats)} beats")

# 3. RR Interval Features (Heart Rate Variability)

In [ ]:
# Extract RR interval features
rr_extractor = RRIntervalFeatures(sampling_rate=record.sampling_rate)
rr_features = rr_extractor.extract_all_features(r_peaks)

print("RR Interval Features:")
print("=" * 40)
for key, value in rr_features.items():
    if isinstance(value, (int, float)):
        print(f"{key:30}: {value:.3f}")

# Visualize RR intervals
rr_intervals = np.diff(r_peaks) / record.sampling_rate

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# RR interval tachogram
axes[0,0].plot(rr_intervals, 'b-o', markersize=3, linewidth=1)
axes[0,0].set_xlabel('Beat Number')
axes[0,0].set_ylabel('RR Interval (s)')
axes[0,0].set_title('RR Interval Tachogram')
axes[0,0].grid(True, alpha=0.3)

# Histogram
axes[0,1].hist(rr_intervals, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0,1].set_xlabel('RR Interval (s)')
axes[0,1].set_ylabel('Frequency')
axes[0,1].set_title('RR Interval Distribution')
axes[0,1].grid(True, alpha=0.3)

# Poincaré plot
rr_n = rr_intervals[:-1]
rr_n1 = rr_intervals[1:]
axes[1,0].scatter(rr_n, rr_n1, alpha=0.6, s=20)
axes[1,0].set_xlabel('RR_n (s)')
axes[1,0].set_ylabel('RR_{n+1} (s)')
axes[1,0].set_title('Poincaré Plot')
axes[1,0].grid(True, alpha=0.3)

# Power spectrum
from scipy.signal import welch
freqs, psd = welch(rr_intervals, fs=4, nperseg=min(128, len(rr_intervals)))
axes[1,1].semilogy(freqs, psd, 'b-', linewidth=1)
axes[1,1].set_xlabel('Frequency (Hz)')
axes[1,1].set_ylabel('Power Spectral Density')
axes[1,1].set_title('RR Interval Power Spectrum')
axes[1,1].grid(True, alpha=0.3)
axes[1,1].set_xlim(0, 0.5)

plt.tight_layout()
plt.show()

# 4. Wavelet Features

In [ ]:
# Extract wavelet features from a sample beat
sample_beat = beats[0].signal
wavelet_extractor = WaveletFeatures(wavelet='db6', level=4)
wavelet_features = wavelet_extractor.extract_all_features(sample_beat)

print("Wavelet Features (sample beat):")
print("=" * 40)
for key, value in list(wavelet_features.items())[:15]:
    print(f"{key:40}: {value:.4f}")

# Visualize wavelet decomposition
coeffs = wavelet_extractor.decompose_signal(sample_beat)
fig, axes = plt.subplots(len(coeffs) + 1, 1, figsize=(14, 10))

# Original signal
axes[0].plot(sample_beat, 'b-', linewidth=1)
axes[0].set_title('Original Beat Signal')
axes[0].grid(True, alpha=0.3)

# Wavelet coefficients
for i, coeff in enumerate(coeffs):
    if i == len(coeffs) - 1:
        title = f'Approximation Coefficients (Level {len(coeffs)-1})'
    else:
        title = f'Detail Coefficients (Level {i+1})'
    axes[i+1].plot(coeff, 'r-', linewidth=1)
    axes[i+1].set_title(title)
    axes[i+1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 5. Morphological Features (PQRST)

In [ ]:
# Extract morphological features
morph_extractor = MorphologicalFeatures(sampling_rate=record.sampling_rate)
r_peak_idx = beats[0].r_peak_index
morph_features = morph_extractor.extract_all_features(sample_beat, r_peak_idx)

print("Morphological Features:")
print("=" * 40)
for key, value in morph_features.items():
    print(f"{key:30}: {value:.3f}")

# Visualize PQRST detection
detection = morph_extractor.detect_pqrst_peaks(sample_beat, r_peak_idx)
peaks = detection['peaks']
amplitudes = detection['amplitudes']

fig, ax = plt.subplots(figsize=(12, 5))
time_ms = np.arange(len(sample_beat)) / record.sampling_rate * 1000

ax.plot(time_ms, sample_beat, 'b-', linewidth=1.5, label='ECG Beat')

# Mark PQRST points
markers = [
    ('P', peaks['p_idx'], amplitudes['p_amp'], 'green'),
    ('Q', peaks['q_idx'], amplitudes['q_amp'], 'yellow'),
    ('R', peaks['r_idx'], amplitudes['r_amp'], 'red'),
    ('S', peaks['s_idx'], amplitudes['s_amp'], 'magenta'),
    ('T', peaks['t_idx'], amplitudes['t_amp'], 'cyan')
]

for label, idx, amp, color in markers:
    if idx and idx > 0:
        ax.scatter(time_ms[idx], amp, c=color, s=100, zorder=5)
        ax.annotate(label, (time_ms[idx], amp), fontsize=12, fontweight='bold',
                   xytext=(5, 5), textcoords='offset points')

# Highlight intervals
if peaks['qrs_onset'] and peaks['qrs_offset']:
    ax.axvspan(time_ms[peaks['qrs_onset']], time_ms[peaks['qrs_offset']],
              alpha=0.2, color='red', label='QRS Complex')
if peaks['p_onset'] and peaks['p_offset']:
    ax.axvspan(time_ms[peaks['p_onset']], time_ms[peaks['p_offset']],
              alpha=0.2, color='green', label='P Wave')
if peaks['t_onset'] and peaks['t_offset']:
    ax.axvspan(time_ms[peaks['t_onset']], time_ms[peaks['t_offset']],
              alpha=0.2, color='orange', label='T Wave')

ax.set_xlabel('Time (ms)')
ax.set_ylabel('Amplitude (mV)')
ax.set_title('PQRST Detection and Annotation')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 6. Feature Fusion

In [ ]:
# Combine all features for multiple beats
fusion = FeatureFusion(
    use_rr_features=True,
    use_wavelet_features=True,
    use_morphological_features=True
)

# Extract features for first 10 beats
all_features = []
for i, beat in enumerate(beats[:10]):
    features = fusion.extract_all_features(
        r_peaks=r_peaks[:i+2] if i+2 <= len(r_peaks) else r_peaks,
        beat=beat.signal,
        r_peak_idx=beat.r_peak_index,
        heart_rate=60 / np.mean(np.diff(r_peaks[:i+2]) / record.sampling_rate) if i+2 <= len(r_peaks) else 75
    )
    feature_vector = fusion.features_to_vector(features)
    all_features.append(feature_vector)

all_features = np.array(all_features)
print(f"Feature matrix shape: {all_features.shape}")

# Visualize feature correlations
corr_matrix = np.corrcoef(all_features[:10, :50].T)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
ax.set_title('Feature Correlation Matrix (first 50 features)')
ax.set_xlabel('Feature Index')
ax.set_ylabel('Feature Index')
plt.colorbar(im, ax=ax, label='Correlation')
plt.tight_layout()
plt.show()

# 7. Feature Importance Analysis

In [ ]:
# Generate synthetic labels for demonstration
np.random.seed(42)
labels = np.random.randint(0, 5, len(all_features))

# Calculate mutual information
from sklearn.feature_selection import mutual_info_classif
mi_scores = mutual_info_classif(all_features, labels)

# Get top features
top_k = 20
top_indices = np.argsort(mi_scores)[-top_k:]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(top_k), mi_scores[top_indices])
ax.set_yticks(range(top_k))
ax.set_yticklabels([f'Feature {i}' for i in top_indices])
ax.set_xlabel('Mutual Information Score')
ax.set_title(f'Top {top_k} Most Informative Features')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()